```{contents}
```

## **Transformer Decoder Layer**

---

### Purpose of the Decoder

The **decoder** is the **generation component** of the Transformer.
It takes:

* **Encoded representations** from the encoder (context), and
* **Previously generated tokens**,

and produces the **next token** in the output sequence (auto-regressively).

---

### Example

In **English → French translation**:

* Encoder processes: `"I love cats"`
* Decoder generates: `"J' aime les chats"`

The decoder:

* Sees encoder outputs (context),
* Predicts one word at a time: `"J'" → "aime" → "les" → "chats"`

---

### Structure of One Decoder Layer

Each decoder layer contains **three sublayers**:

```
             ┌───────────────────────────────┐
Input  --->  │ 1. Masked Multi-Head Attention │
             └───────────────────────────────┘
                        │
                        ▼
             ┌───────────────────────────────┐
             │ 2. Encoder–Decoder Attention   │
             └───────────────────────────────┘
                        │
                        ▼
             ┌───────────────────────────────┐
             │ 3. Feed Forward Network (FFN)  │
             └───────────────────────────────┘
                        │
                        ▼
                   Add + LayerNorm after each sublayer
```

---

### Step-by-Step Workflow

Let’s denote
* $X_{dec}$: input embeddings (decoder input sequence)
* $E_{enc}$: encoder outputs
* $d_{model} = 512$

---

### **Step 1: Masked Multi-Head Self-Attention**

The decoder first applies **self-attention** — but with a **mask**.

#### Why Masking?

The model generates tokens one-by-one during training.
Each position **must not attend to future positions**.

✅ Example:
When predicting the 3rd word, it should only see words 1 and 2 — **not** 4 or 5.

So, we use a **look-ahead mask** (upper-triangular matrix) to zero out attention beyond the current position.

#### Formula:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + \text{mask}\right)V
$$

Where masked entries are set to $-\infty$, so softmax = 0.

#### Output:

$$
Z_1 = \text{MaskedMultiHead}(X_{dec}, X_{dec}, X_{dec})
$$

Then apply residual + layer norm:
$$
X_1 = \text{LayerNorm}(X_{dec} + Z_1)
$$

---

### 🔹 **Step 2: Encoder–Decoder Attention (Cross-Attention)**

This is what connects the **encoder** and **decoder**.

Here:

* Queries $Q$ come from the **decoder** (output of Step 1).
* Keys $K$, Values $V$ come from the **encoder outputs**.

So the decoder “looks at” the entire encoded input sequence to decide what to generate next.

#### Formula:

$$
Z_2 = \text{MultiHead}(Q = X_1, K = E_{enc}, V = E_{enc})
$$

Then apply residual + layer norm:
$$
X_2 = \text{LayerNorm}(X_1 + Z_2)
$$

---

### **Step 3: Feed-Forward Network (FFN)**

Each token position is processed independently through a two-layer feedforward network:

$$
\text{FFN}(x) = \max(0, xW_1 + b_1)W_2 + b_2
$$

* Expands from $d_{model}=512$ → $d_{ff}=2048$
* Then projects back to 512

Then apply residual + layer norm again:
$$
Y = \text{LayerNorm}(X_2 + \text{FFN}(X_2))
$$

---

### Mathematical Summary

| Sub-layer | Operation                 | Formula                                                 |
| --------- | ------------------------- | ------------------------------------------------------- |
| 1         | Masked Self-Attention     | $X_1 = \text{LayerNorm}(X + \text{MHA}(X, X, X))$     |
| 2         | Encoder–Decoder Attention | $X_2 = \text{LayerNorm}(X_1 + \text{MHA}(X_1, E, E))$ |
| 3         | Feed Forward              | $Y = \text{LayerNorm}(X_2 + \text{FFN}(X_2))$         |

Where MHA = Multi-Head Attention.

---

### Intuition Behind Each Component

| Component                 | Role                                         | Analogy                                               |
| ------------------------- | -------------------------------------------- | ----------------------------------------------------- |
| **Masked Self-Attention** | Understand partial sentence generated so far | “What have I said already?”                           |
| **Cross-Attention**       | Attend to relevant encoder words             | “Which input words are related to my current word?”   |
| **FFN**                   | Non-linear transformation per token          | “Adjust or refine meaning before output”              |
| **Residual + LayerNorm**  | Stabilize training                           | “Keep gradients healthy and representations balanced” |

---

### Visualization of Attention Flow

```
Encoder Outputs (context vectors)
          │
          ▼
 ┌──────────────────────────────┐
 │ Encoder–Decoder Attention    │ ← Decoder uses context here
 └──────────────────────────────┘
          ▲
          │
 ┌──────────────────────────────┐
 │ Masked Self-Attention (past) │ ← Only previous tokens visible
 └──────────────────────────────┘
          │
          ▼
    Feed Forward Network
          │
          ▼
    Output Token Probabilities
```

---

### Decoder Stack

There are **N = 6 decoder layers**, each processing the output of the previous one.
Final layer output goes to a **linear layer + softmax** to predict token probabilities.

$$
P(y_t) = \text{softmax}(W_o Y_t)
$$

---

### Auto-Regressive Generation (Inference)

During inference, the decoder works **one step at a time**:

1. Input previous predicted tokens
2. Apply all decoder layers
3. Generate next token (argmax or sampling)
4. Append token and repeat

Masked self-attention ensures it only attends to **past tokens** at every step.

---

### Visualization (Conceptual Flow)

```
Input: "I love cats"
Encoder → Context vectors
                    ↓
Decoder Input: <sos>
Masked Self-Attention → "J'"
Cross-Attention → Encoder info → "J'aime"
Feed-Forward → next word
Repeat → "J'aime les chats"
```

---

**Summary Table**

| Stage | Attention Type            | Source of Q, K, V                | Masking | Purpose                     |
| ----- | ------------------------- | -------------------------------- | ------- | --------------------------- |
| 1     | Masked Self-Attention     | Q=K=V from decoder               | ✅ Yes   | Understand previous outputs |
| 2     | Encoder–Decoder Attention | Q from decoder, K,V from encoder | ❌ No    | Use source context          |
| 3     | FFN                       | Independent per position         | ❌ No    | Non-linear refinement       |

---

**In Essence**

> The Transformer **decoder** learns to generate text one token at a time,
> looking backward (via masked self-attention) to understand what’s been generated,
> and looking sideways (via cross-attention) to see what the input means.
> Residual connections and layer normalization keep everything smooth and stable.

